In [1]:
import os
import requests
import json
from openai import OpenAI
import pandas as pd

In [2]:
!pip install zhipuai -q

读取密钥

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ZHIPU_API_KEY = user_secrets.get_secret("ZHIPU_API_KEY")
print(f"DEBUG: Key successfully loaded, prefix={ZHIPU_API_KEY[:10]}****")
client = OpenAI(
    api_key=ZHIPU_API_KEY,
    base_url="https://open.bigmodel.cn/api/paas/v4/"
)


DEBUG: Key successfully loaded, prefix=8cbf1c3e53****


读取Skill文件

In [4]:
with open("/kaggle/working/SKILL.md", "r", encoding="utf-8") as f:
    skill_content = f.read()

世界银行API工具函数

多轮对话收集业务信息模拟智能体追问逻辑

In [5]:
WB_BASE_URL = "http://api.worldbank.org/v2/country/{country_code}/indicator/{indicator}?format=json&per_page=1"

def get_wb_indicator(country_code: str, indicator_code: str):
    url = WB_BASE_URL.format(country_code=country_code, indicator=indicator_code)
    resp = requests.get(url, timeout=15)
    if resp.status_code !=200:
        return None
    data = resp.json()
    if len(data)<2 or len(data[1])==0:
        return None
    entry = data[1][0]
    value = entry["value"]
    year = entry["date"]
    if value is None:
        return {"value": None, "year": year}
    try:
        value = float(value)
    except:
        pass
    return {"value":value, "year":year}

wb_indicators = [
    "NY.GDP.MKTP.CD",
    "NY.GDP.PCAP.CD",
    "NY.GDP.MKTP.KD.ZG",
    "FP.CPI.TOTL.ZG",
    "SP.POP.TOTL",
    "SP.URB.TOTL.IN.ZS",
    "IT.NET.USER.ZS",
    "NE.IMP.GNFS.ZS",
    "SL.UEM.TOTL.ZS"
]

In [6]:
class OverseaAgent:
    def __init__(self):
        # 需要收集的7个必填字段 + 对应的提问话术
        self.questions_map = {
            "product": "1.请输入你的出海产品类型：",
            "country_codes": "2.请输入候选国家三位ISO代码，多个用英文逗号隔开，例如 GBR,USA,JPN,SAU：",
            "target_customer": "3.请描述目标客群：",
            "price_range": "4.请输入海外终端价格区间，例如80‑150美元：",
            "enter_time": "5.期望市场进入时间周期（例如6个月快速切入 / 1‑2年稳步布局）：",
            "channel_resource": "6.公司现有海外渠道资源：",
            "risk_appetite": "7.风险偏好（低风险 / 中等风险 / 高风险）："
        }
        self.collected = {k: None for k in self.questions_map.keys()}

    def check_all_filled(self):
        """判断7项信息是否全部收集完毕"""
        return all(v is not None for v in self.collected.values())

    def interactive_gather_info(self):
        """交互式循环，逐个向用户提问，收集全部业务信息"""
        print("\n=====出海智能体已启动=====")
        trigger = input("请输入指令（输入：开始出海分析 启动分析）：").strip()
        if trigger != "开始出海分析":
            print("未识别触发指令，请重新运行cell，输入：开始出海分析")
            return False

        print("\n【智能体】收到出海分析请求，现在需要向你确认7项关键业务信息\n")
        for key, prompt_text in self.questions_map.items():
            ans = input(prompt_text).strip()
            if key == "country_codes":
                # 国家字段转列表
                self.collected[key] = [x.strip() for x in ans.split(",")]
            else:
                self.collected[key] = ans
        print("\n✅全部业务信息收集完成！")
        print(json.dumps(self.collected, ensure_ascii=False, indent=2))
        return True

agent = OverseaAgent()

业务参数

In [7]:
ok = agent.interactive_gather_info()
if not ok or not agent.check_all_filled():
    raise SystemExit("信息收集不完整，程序终止，请重新运行单元格")


=====出海智能体已启动=====


请输入指令（输入：开始出海分析 启动分析）： 开始出海分析



【智能体】收到出海分析请求，现在需要向你确认7项关键业务信息



1.请输入你的出海产品类型： 智能煎烤机/小家电
2.请输入候选国家三位ISO代码，多个用英文逗号隔开，例如 GBR,USA,JPN,SAU： GBR,USA,JPN,SAU
3.请描述目标客群： 厨房小白、中产家庭
4.请输入海外终端价格区间，例如80‑150美元： 120美元
5.期望市场进入时间周期（例如6个月快速切入 / 1‑2年稳步布局）： 6个月快速切入
6.公司现有海外渠道资源： 海外本地经销商
7.风险偏好（低风险 / 中等风险 / 高风险）： 中等风险



✅全部业务信息收集完成！
{
  "product": "智能煎烤机/小家电",
  "country_codes": [
    "GBR",
    "USA",
    "JPN",
    "SAU"
  ],
  "target_customer": "厨房小白、中产家庭",
  "price_range": "120美元",
  "enter_time": "6个月快速切入",
  "channel_resource": "海外本地经销商",
  "risk_appetite": "中等风险"
}


批量拉取世界银行全部国家指标数据

In [8]:
all_country_data = {}
countries = agent.collected["country_codes"]
for cc in countries:
    res = {}
    for ind in wb_indicators:
        d = get_wb_indicator(cc, ind)
        res[ind] = d
    all_country_data[cc] = res

print("\n====智能体自动获取世界银行原始宏观数据完成====")
for c_code,dat in all_country_data.items():
    print(f"\n国家 {c_code}:")
    for k,v in dat.items():
        if v is None:
            print(f"  {k} : 数据缺失")
        else:
            print(f"  {k} : value={v['value']}, year={v['year']}")


====智能体自动获取世界银行原始宏观数据完成====

国家 GBR:
  NY.GDP.MKTP.CD : value=4002587541846.01, year=2025
  NY.GDP.PCAP.CD : value=57601.9621201954, year=2025
  NY.GDP.MKTP.KD.ZG : value=1.38844207858726, year=2025
  FP.CPI.TOTL.ZG : value=3.88306881626, year=2025
  SP.POP.TOTL : value=69487000.0, year=2025
  SP.URB.TOTL.IN.ZS : value=83.323608078271, year=2025
  IT.NET.USER.ZS : value=None, year=2025
  NE.IMP.GNFS.ZS : value=31.8603712472379, year=2025
  SL.UEM.TOTL.ZS : value=4.746, year=2025

国家 USA:
  NY.GDP.MKTP.CD : value=30769700000000.0, year=2025
  NY.GDP.PCAP.CD : value=90026.5163005744, year=2025
  NY.GDP.MKTP.KD.ZG : value=2.16138195623856, year=2025
  FP.CPI.TOTL.ZG : value=None, year=2025
  SP.POP.TOTL : value=341784857.0, year=2025
  SP.URB.TOTL.IN.ZS : value=80.1921733027869, year=2025
  IT.NET.USER.ZS : value=None, year=2025
  NE.IMP.GNFS.ZS : value=None, year=2025
  SL.UEM.TOTL.ZS : value=4.198, year=2025

国家 JPN:
  NY.GDP.MKTP.CD : value=4435162999976.94, year=2025
  NY.GDP.PCAP.CD

组装prompt调用DeepSeek，加载skill生成完整出海报告

In [9]:
system_prompt = f"""
你是出海战略分析智能体，严格执行SKILL.md的overseas‑feasibility技能。
# SKILL文档
{skill_content}

## 用户交互得到业务参数
{json.dumps(agent.collected,ensure_ascii=False,indent=2)}

## 智能体调用世界银行API获取宏观数据；value=None代表数据缺失，报告标注“数据缺失”，禁止编造
{json.dumps(all_country_data,ensure_ascii=False,indent=2)}

任务：
1.使用SKILL规定6个维度、权重，每个国家计算0‑100分项得分，计算加权总分；
2.严格遵守SKILL强制输出完整报告结构；
3.写明数据来源World Bank，标注指标对应年份；
4.选出加权总分最高国家作为优先目标市场，输出至少3条基于数据的推荐理由；
5.输出下一步行动清单；
6.末尾强制声明：本报告仅基于公开宏观数据，不构成投资建议。
"""

user_prompt = "执行overseas‑feasibility技能，输出完整出海可行性商务决策报告"

resp = client.chat.completions.create(
    model="glm-4-flash",
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_prompt}
    ],
    temperature=0.3
)

final_report = resp.choices[0].message.content

print("\n"+"="*80)
print("【出海可行性决策报告】")
print(final_report)

# 智能体自动保存报告文件
with open("/kaggle/working/出海可行性报告.md","w",encoding="utf-8") as f:
    f.write(final_report)
print("\n✅报告自动保存至 /kaggle/working/出海可行性报告.md")


【出海可行性决策报告】
# 出海市场可行性分析报告

## 1. 业务需求概述
- 产品：智能煎烤机/小家电
- 候选国家：GBR（英国）、USA（美国）、JPN（日本）、SAU（沙特阿拉伯）
- 目标消费群体：厨房小白、中产家庭
- 海外终端售价区间：120美元
- 期望市场进入周期：6个月快速切入
- 企业现有海外渠道资源：海外本地经销商
- 企业风险偏好：中等风险

## 2. 候选国家宏观数据一览

| 国家 | GDP总量（美元） | 人均GDP（美元） | GDP年度增长率（%） | CPI通货膨胀率（%） | 总人口 | 城镇化率（%） | 互联网用户占总人口比例（%） | 进口商品服务占GDP比重（%） | 失业率（%） | 数据来源：World Bank World Development Indicators |
| ---- | -------------- | -------------- | ---------------- | ---------------- | ------ | ------------ | ----------------------- | ----------------------- | ---------- | -------------------------------------------------------------- |
| GBR | 4002587541846.01 | 57601.96 | 1.38844 | 3.88307 | 69487000 | 83.32 | None | 31.86 | 4.746 | 2025 |
| USA | 30769700000000.0 | 90026.52 | 2.16138 | None | 341784857 | 80.19 | None | None | 4.198 | 2025 |
| JPN | 4435162999976.94 | 35951.04 | 1.19311 | 3.17253 | 123366734 | 92.30 | None | None | 2.451 | 2025 |
| SAU | 1276942933333.33 | 34536.66 | 4.50247 | 2.08421 | 36973555 | 84.61